# Request a SO-ARM101 (LeRobot)

Provision a Chameleon Edge device for SO-ARM101 robot arm teleoperation and
data collection using HuggingFace LeRobot.

## Prerequisites
- Active Chameleon Cloud allocation (project `CHI-261589`)
- A SO-ARM101 device registered on CHI@Edge
- Container image with LeRobot + teleoperation stack pushed to Docker Hub
- python-chi 1.0+ (`pip install python-chi`)

In [24]:
import chi
from datetime import timedelta

chi.use_site("CHI@Edge")
chi.set("project_name", "CHI-261589")

from chi.lease import Lease
from chi.container import Container

Now using CHI@Edge:
URL: https://chi.edge.chameleoncloud.org
Location: University of Chicago, Chicago, Illinois, USA
Support contact: help@chameleoncloud.org


## Lease the device

Request (or reuse) a lease for the SO-ARM101 edge device. If an active lease
named `LEASE_NAME` already exists it will be reused — safe to re-run.

In [25]:
from chi import lease as lease_api

LEASE_NAME = "lerobot-soarm101-lease"
DEVICE_NAME = "soarm101-1"

# Check for existing active lease before creating
my_lease = None
try:
    existing = lease_api.get_lease(LEASE_NAME)
    if existing and existing.status in ("ACTIVE", "PENDING"):
        my_lease = existing
        print(f"Reusing lease '{my_lease.name}' [{my_lease.status}]")
        print(f"  ID:   {my_lease.id}")
        print(f"  Ends: {my_lease.end_date}")
except Exception:
    pass

if my_lease is None:
    print(f"No active lease named '{LEASE_NAME}'. Creating...")
    # CHI@Edge max lease duration is 7 days
    my_lease = Lease(name=LEASE_NAME, duration=timedelta(days=7))
    my_lease.add_device_reservation(device_name=DEVICE_NAME, amount=1)
    my_lease.submit(wait_for_active=True, idempotent=True)
    print(f"Lease '{my_lease.name}' is {my_lease.status} (id: {my_lease.id})")
    print(f"Ends: {my_lease.end_date}")

Reusing lease 'lerobot-soarm101-lease' [ACTIVE]
  ID:   96d5dd14-c18a-4380-b09d-b4f487bace17
  Ends: 2026-04-11 23:50:00


## Launch the container

Create a container with the LeRobot stack. If an errored or stale container
with the same name exists it will be deleted first. Safe to re-run.

> **Note:** The SO-ARM101 uses USB serial (`ttyACM*`), not hardware UART (`ttyS0`).
> Do not use `pi_serial` or `pi_gpio` device profiles — they request resources
> unavailable on this node and cause a scheduling error.

### Build and push the image first (run on your local machine)

```bash
cd coachable-robots
docker buildx build --platform linux/arm64 \
    -t rianders/lerobot-soarm101:main \
    -f docker/Dockerfile.pi . --push
```

In [ ]:
import time
from chi import clients
from chi.container import Container

CONTAINER_NAME = "lerobot-soarm101-container"

# ── Clean up any stale container with this name ──
zun = clients.zun()
existing = [c for c in zun.containers.list() if c.name == CONTAINER_NAME]
if existing:
    stale = existing[0]
    print(f"Found existing container [{stale.status}] — deleting...")
    stale_container = Container.from_zun_container(stale)
    stale_container.delete()
    for _ in range(12):
        time.sleep(5)
        if not any(c.name == CONTAINER_NAME for c in zun.containers.list()):
            print("  Deleted.")
            break
    else:
        raise RuntimeError("Timed out waiting for stale container to delete.")

# ── Create ──
reservation_id = my_lease.device_reservations[0]["id"]

my_container = Container(
    name=CONTAINER_NAME,
    image_ref="rianders/lerobot-soarm101:main",
    exposed_ports=["22/tcp"],
    reservation_id=reservation_id,
    # SO-ARM101 uses USB serial (ttyACM*) — no device profile needed
)
my_container.submit(wait_for_active=True, idempotent=True)

print(f"Container '{my_container.name}' is {my_container.status}")
print(f"ID: {my_container.zun_container.uuid}")

Found existing container [Error] — deleting...
  Deleted.


In [ ]:
print(my_container.logs())

In [ ]:
print(f"Container '{my_container.name}' is {my_container.status}")

## Assign a floating IP

In [ ]:
# Assign floating IP if not already attached
if not getattr(my_container, 'floating_ip', None):
    my_container.associate_floating_ip()

print(f"Public IP: {my_container.floating_ip}")
print(f"\nSSH: ssh root@{my_container.floating_ip}")

## Verify the environment

Quick checks that the arm, cameras, and LeRobot are accessible.

In [ ]:
output, exit_code = my_container.execute("ls -l")
print(output)

In [ ]:
# Check that the SO-ARM101 serial device is visible
output, _ = my_container.execute("ls -l /dev/ttyUSB* /dev/ttyACM* 2>/dev/null || echo 'No serial devices found'")
print(output)

In [ ]:
# Check for connected cameras
output, _ = my_container.execute("ls -l /dev/video* 2>/dev/null || echo 'No video devices found'")
print(output)

In [ ]:
# Verify Python and LeRobot are installed
output, _ = my_container.execute("python3 --version && python3 -c 'import lerobot; print(f\"LeRobot version: {lerobot.__version__}\")'")
print(output)

## Teleoperation & Data Collection

Use the LeRobot v0.4.1 CLI (`lerobot-record`) to control the arm and
record demonstration episodes into a LeRobot-compatible dataset.

### Supported policies for training later
- **ACT** — recommended for MI100 (no Flash Attention needed)
- **Diffusion** — heavier compute, solid results
- **VQ-BeT** — good for multi-modal action distributions
- **Pi0Fast / Pi0.5** — VLA models, need gradient checkpointing on MI100
- **SmolVLA** — lightweight VLA for constrained hardware

In [ ]:
# Teleoperate the arm (no recording, just control)
# TODO: update camera config to match your setup
teleop_cmd = """\
lerobot-teleoperate \
    --robot.type=so101_follower \
    --robot.cameras='{ top: {type: opencv, index_or_path: 0, width: 640, height: 480, fps: 30} }' \
    --teleop.type=so101_leader
"""
print("Starting teleoperation ...")
output, _ = my_container.execute(f"bash -c '{teleop_cmd.strip()} > /tmp/teleop.log 2>&1 &'")
print(output)

In [ ]:
# Record demonstration episodes to a LeRobot dataset
# TODO: adjust repo_id, num_episodes, and camera config
record_cmd = """\
lerobot-record \
    --robot.type=so101_follower \
    --robot.cameras='{ top: {type: opencv, index_or_path: 0, width: 640, height: 480, fps: 30} }' \
    --teleop.type=so101_leader \
    --dataset.repo_id=rianders/soarm101-episodes \
    --dataset.num_episodes=10 \
    --dataset.push_to_hub=true
"""
print("Starting data collection ...")
output, _ = my_container.execute(f"bash -c '{record_cmd.strip()}'")
print(output)

In [ ]:
# List collected dataset files
output, _ = my_container.execute("find /lerobot/data -type f | head -30")
print(output)

## Cleanup

Destroy the container and release the lease when finished.

In [ ]:
import time
from chi import clients

confirm = input(f"Delete container '{CONTAINER_NAME}'? [y/N]: ")
if confirm.strip().lower() in ('y', 'yes'):
    zun = clients.zun()

    # Find the container
    containers = zun.containers.list()
    target = next((c for c in containers if c.name == CONTAINER_NAME), None)

    if target is None:
        print(f"No container named '{CONTAINER_NAME}' found — already deleted?")
    else:
        print(f"Deleting {target.name} ({target.uuid}) [{target.status}]...")
        my_container.delete()

        # Poll until gone
        for i in range(12):
            time.sleep(5)
            remaining = [c for c in zun.containers.list() if c.name == CONTAINER_NAME]
            if not remaining:
                print(f"Confirmed deleted after {(i+1)*5}s.")
                break
            print(f"  {(i+1)*5}s... still {remaining[0].status}")
        else:
            print("Timed out waiting for deletion — check the portal.")
else:
    print("Cancelled.")

In [ ]:
# Remove lease when you're done with the device
# my_lease.delete()